# Tessera Portable Validation Console

Run the launcher cell once. It starts a server inside the active Colab runtime and prints a session-only URL with an **Open Standalone Console** link. The interface opens in a separate browser tab (not an embedded notebook frame). The URL expires when the runtime stops.

Use only trusted repositories and commands. Colab Secrets are not isolated from arbitrary code running in the same runtime. Linux mock tests do not validate native Windows hook behavior.


In [ ]:
#@title Tessera Validation Server · Launch Console { display-mode: "form" }
import os, re, json, time, html, shutil, socket, secrets, pathlib, tempfile, threading, subprocess, datetime, platform
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlsplit
from IPython.display import display, HTML

try:
    from google.colab import userdata, output as colab_output
except Exception:
    userdata = None
    colab_output = None

# One local server per active Colab runtime. The secret path is ephemeral.
APP_SECRET = secrets.token_urlsafe(24)
BASE_PATH = "/tessera/" + APP_SECRET + "/"
WORKSPACE = pathlib.Path("/content/tessera-validation-workspace")
REPORT_PATH = pathlib.Path("/content/PROJECT_VALIDATION_REPORT.txt")
LOCK = threading.RLock()
STATE = {
    "running": False, "stage": "Idle", "overall": "IDLE",
    "logs": [], "passed": 0, "failed": 0, "skipped": 0,
    "report_ready": False, "started": None, "duration": 0
}

def safe_clip(value, limit=16000):
    value = str(value or "")
    return value if len(value) <= limit else value[:limit//2] + "\n...[OUTPUT TRIMMED]...\n" + value[-limit//2:]

def redact(value, token):
    value = str(value or "")
    return value.replace(token, "[REDACTED]") if token else value

def update_state(**values):
    with LOCK:
        STATE.update(values)

def append_log(name, status, command="", output="", duration=0, token=None):
    entry = {
        "name": str(name), "status": str(status),
        "command": safe_clip(redact(command, token), 3000),
        "output": safe_clip(redact(output, token)),
        "duration": round(float(duration or 0), 2)
    }
    with LOCK:
        STATE["logs"].append(entry)
        STATE["stage"] = str(name)
    return entry

def clean_test_environment():
    env = os.environ.copy()
    sensitive_markers = ("TOKEN", "PASSWORD", "SECRET", "CREDENTIAL", "AUTH")
    for key in list(env):
        if any(marker in key.upper() for marker in sensitive_markers):
            env.pop(key, None)
    for key in ("GIT_ASKPASS", "SSH_ASKPASS", "TESSERA_GIT_USER", "TESSERA_GIT_TOKEN"):
        env.pop(key, None)
    env["GIT_TERMINAL_PROMPT"] = "0"
    return env

def execute(command, cwd, env, timeout):
    started = time.monotonic()
    try:
        proc = subprocess.run(
            command, cwd=str(cwd), env=env, shell=True, executable="/bin/bash",
            text=True, capture_output=True, timeout=timeout
        )
        output = (proc.stdout or "") + (("\n[STDERR]\n" + proc.stderr) if proc.stderr else "")
        return proc.returncode, time.monotonic() - started, output
    except subprocess.TimeoutExpired as exc:
        out = exc.stdout or ""
        err = exc.stderr or ""
        if isinstance(out, bytes): out = out.decode("utf-8", "replace")
        if isinstance(err, bytes): err = err.decode("utf-8", "replace")
        return 124, time.monotonic() - started, "TIMEOUT after " + str(timeout) + "s\n" + out + "\n" + err
    except Exception as exc:
        return -1, time.monotonic() - started, type(exc).__name__ + ": " + str(exc)

def run_validation(cfg):
    start = time.monotonic()
    issues, token = [], None
    setup_results, test_results = [], []
    cloned, commit, setup_blocked = False, "unknown", False
    keep_workspace = bool(cfg.get("keep_workspace"))
    try:
        repo = str(cfg.get("repo", "")).strip()
        branch = str(cfg.get("ref", "")).strip()
        secret_name = str(cfg.get("secret_name", "GH_TOKEN")).strip()
        git_user = str(cfg.get("git_user", "x-access-token")).strip() or "x-access-token"
        clone_timeout = int(cfg.get("clone_timeout", 240))
        test_timeout = int(cfg.get("test_timeout", 900))
        depth = int(cfg.get("depth", 1))
        continue_on_error = bool(cfg.get("continue_on_error", True))
        setup_commands = [str(x).strip() for x in cfg.get("setup", []) if str(x).strip()]
        tests = cfg.get("tests", [])
        if not re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", repo):
            raise ValueError("Use a GitHub HTTPS repository URL without embedded credentials.")
        if not 1 <= clone_timeout <= 3600 or not 1 <= test_timeout <= 7200 or not 0 <= depth <= 10000:
            raise ValueError("Timeout/depth value is outside the supported range.")
        if not isinstance(tests, list) or not tests:
            raise ValueError("Add at least one test.")
        normalized = []
        for index, item in enumerate(tests, 1):
            name = str(item.get("name", "")).strip()
            command = str(item.get("command", "")).strip()
            needs = [x.strip() for x in str(item.get("needs", "")).split(",") if x.strip()]
            if not name or not command:
                raise ValueError("Test row " + str(index) + " needs both a name and a command.")
            normalized.append({"name": name, "command": command, "needs": needs})
        update_state(running=True, stage="Preparing checkout", overall="RUNNING",
                     logs=[], passed=0, failed=0, skipped=0, report_ready=False,
                     started=time.time(), duration=0)
        shutil.rmtree(WORKSPACE, ignore_errors=True)
        clone_env = os.environ.copy()
        clone_env["GIT_TERMINAL_PROMPT"] = "0"
        clone_env["GCM_INTERACTIVE"] = "Never"
        ask_dir = None
        try:
            if secret_name and userdata is not None:
                try:
                    token = userdata.get(secret_name)
                except Exception as exc:
                    append_log("GitHub Secret", "WARN", output="Secret unavailable; trying anonymous clone. " + type(exc).__name__)
            if token:
                ask_dir = pathlib.Path(tempfile.mkdtemp(prefix="tessera-askpass-"))
                ask_script = ask_dir / "askpass.sh"
                ask_script.write_text(
                    '#!/bin/sh\ncase "$1" in *Username*) printf "%s\\n" "$TESSERA_GIT_USER";; *) printf "%s\\n" "$TESSERA_GIT_TOKEN";; esac\n',
                    encoding="utf-8"
                )
                ask_script.chmod(0o700)
                clone_env.update({
                    "GIT_ASKPASS": str(ask_script),
                    "TESSERA_GIT_USER": git_user,
                    "TESSERA_GIT_TOKEN": token
                })
            clone_cmd = ["git", "-c", "credential.helper=", "clone"]
            if depth:
                clone_cmd += ["--depth", str(depth)]
            if branch:
                clone_cmd += ["--single-branch", "--branch", branch]
            clone_cmd += [repo, str(WORKSPACE)]
            update_state(stage="Cloning repository")
            t0 = time.monotonic()
            try:
                cp = subprocess.run(clone_cmd, env=clone_env, text=True, capture_output=True, timeout=clone_timeout)
                clone_output = (cp.stdout or "") + (("\n[STDERR]\n" + cp.stderr) if cp.stderr else "")
                clone_code = cp.returncode
            except subprocess.TimeoutExpired:
                clone_code, clone_output = 124, "Clone timed out after " + str(clone_timeout) + " seconds."
            if clone_code == 0:
                cloned = True
                commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKSPACE, text=True).strip()
                append_log("Repository checkout", "PASS", "git clone", "Checked out " + commit, time.monotonic()-t0, token)
            else:
                issues.append(("Repository checkout", redact(safe_clip(clone_output), token)))
                append_log("Repository checkout", "FAIL", "git clone", clone_output, time.monotonic()-t0, token)
        finally:
            if ask_dir:
                shutil.rmtree(ask_dir, ignore_errors=True)
            for key in ("GIT_ASKPASS", "TESSERA_GIT_USER", "TESSERA_GIT_TOKEN"):
                clone_env.pop(key, None)

        safe_token = token
        token = None
        test_env = clean_test_environment()
        if cloned:
            for index, command in enumerate(setup_commands, 1):
                update_state(stage="Setup " + str(index) + "/" + str(len(setup_commands)))
                code, duration, output_text = execute(command, WORKSPACE, test_env, test_timeout)
                status = "PASS" if code == 0 else "FAIL"
                append_log("Setup " + str(index), status, command, output_text, duration, safe_token)
                setup_results.append({"name": "Setup " + str(index), "status": status, "command": command, "output": redact(output_text, safe_token), "duration": duration, "exit": code})
                if code != 0:
                    issues.append(("Setup " + str(index), redact(safe_clip(output_text), safe_token) or "exit=" + str(code)))
                    setup_blocked = True
                    if not continue_on_error:
                        break
            if setup_blocked:
                for spec in normalized:
                    test_results.append({"name": spec["name"], "status": "SKIP", "command": spec["command"], "output": "Skipped because a setup command failed.", "duration": 0, "exit": None})
                    append_log(spec["name"], "SKIP", spec["command"], "Skipped because a setup command failed.")
            else:
                stop_after_failure = False
                for spec in normalized:
                    if stop_after_failure:
                        test_results.append({"name": spec["name"], "status": "SKIP", "command": spec["command"], "output": "Skipped because continue-after-failure is disabled.", "duration": 0, "exit": None})
                        append_log(spec["name"], "SKIP", spec["command"], "Skipped because continue-after-failure is disabled.")
                        continue
                    missing = [tool for tool in spec["needs"] if shutil.which(tool) is None]
                    if missing:
                        test_results.append({"name": spec["name"], "status": "SKIP", "command": spec["command"], "output": "Missing executables: " + ", ".join(missing), "duration": 0, "exit": None})
                        append_log(spec["name"], "SKIP", spec["command"], "Missing executables: " + ", ".join(missing))
                        continue
                    update_state(stage="Test: " + spec["name"])
                    code, duration, output_text = execute(spec["command"], WORKSPACE, test_env, test_timeout)
                    status = "PASS" if code == 0 else "FAIL"
                    test_results.append({"name": spec["name"], "status": status, "command": spec["command"], "output": redact(output_text, safe_token), "duration": duration, "exit": code})
                    append_log(spec["name"], status, spec["command"], output_text, duration, safe_token)
                    if code != 0:
                        issues.append((spec["name"], redact(safe_clip(output_text), safe_token) or "exit=" + str(code)))
                        if not continue_on_error:
                            stop_after_failure = True
        else:
            for spec in normalized:
                test_results.append({"name": spec["name"], "status": "SKIP", "command": spec["command"], "output": "Skipped because repository checkout failed.", "duration": 0, "exit": None})
                append_log(spec["name"], "SKIP", spec["command"], "Skipped because repository checkout failed.")

        passed = sum(x["status"] == "PASS" for x in test_results)
        failed = sum(x["status"] == "FAIL" for x in test_results)
        skipped = sum(x["status"] == "SKIP" for x in test_results)
        overall = "FAIL" if issues or failed or not cloned else ("PARTIAL" if skipped else ("PASS" if passed else "FAIL"))
        elapsed = round(time.monotonic() - start, 2)
        report_lines = [
            "TESSERA PORTABLE VALIDATION REPORT", "Overall: " + overall,
            "Repository: " + repo, "Ref: " + (branch or "(default)"),
            "Commit: " + commit, "Timestamp UTC: " + datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "Environment: " + platform.platform() + " | Python " + platform.python_version(),
            "Duration: " + str(elapsed) + "s",
            "Results: " + str(passed) + " PASS / " + str(failed) + " FAIL / " + str(skipped) + " SKIP", "",
            "SETUP:"
        ]
        if not setup_results:
            report_lines.append("(none)")
        for item in setup_results:
            report_lines.extend(["[" + item["status"] + "] " + item["name"] + " | exit=" + str(item["exit"]),
                                 "COMMAND: " + redact(item["command"], safe_token), safe_clip(item["output"]), ""])
        report_lines.append("TESTS:")
        for item in test_results:
            report_lines.extend(["[" + item["status"] + "] " + item["name"] + " | exit=" + str(item["exit"]),
                                 "COMMAND: " + redact(item["command"], safe_token), safe_clip(item["output"]), ""])
        report_lines.append("ISSUES:")
        if not issues:
            report_lines.append("(none)")
        for title, detail in issues:
            report_lines.extend(["--- " + title + " ---", detail, ""])
        report = "\n".join(report_lines)
        REPORT_PATH.write_text(report, encoding="utf-8")
        update_state(running=False, stage="Completed", overall=overall,
                     passed=passed, failed=failed, skipped=skipped,
                     report_ready=True, duration=elapsed)
        if cloned and not keep_workspace:
            shutil.rmtree(WORKSPACE, ignore_errors=True)
    except Exception as exc:
        message = type(exc).__name__ + ": " + str(exc)
        append_log("Validation error", "FAIL", output=message, token=token)
        elapsed = round(time.monotonic() - start, 2)
        error_report = "TESSERA PORTABLE VALIDATION REPORT\nOverall: FAIL\nError: " + redact(message, token) + "\nDuration: " + str(elapsed) + "s\n"
        try:
            REPORT_PATH.write_text(error_report, encoding="utf-8")
        except Exception:
            pass
        update_state(running=False, stage="Failed", overall="FAIL",
                     report_ready=REPORT_PATH.exists(), duration=elapsed, failed=1)
        if not keep_workspace:
            shutil.rmtree(WORKSPACE, ignore_errors=True)

PAGE = r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<meta name="color-scheme" content="dark"><title>Tessera · Validation Center</title>
<style>
:root{color-scheme:dark;--bg:#080e19;--panel:#101a2b;--panel2:#142238;--line:#26374e;--text:#edf3fb;--muted:#91a3ba;--teal:#43d6c5;--blue:#81adff;--green:#45d6a0;--red:#ff7185;--amber:#f5bd69}
*{box-sizing:border-box}body{margin:0;background:radial-gradient(ellipse at 12% -10%,#183653 0,transparent 35%),radial-gradient(ellipse at 100% 0,#142542 0,transparent 30%),var(--bg);color:var(--text);font:14px/1.5 Inter,ui-sans-serif,system-ui,-apple-system,Segoe UI,sans-serif;min-height:100vh}
button,input,textarea,select{font:inherit}button{cursor:pointer}.shell{max-width:1440px;margin:auto;padding:24px clamp(14px,3vw,38px) 44px}
.top{display:flex;align-items:center;justify-content:space-between;gap:16px;padding:5px 0 22px;border-bottom:1px solid #ffffff12}
.brand{display:flex;align-items:center;gap:12px}.mark{width:38px;height:38px;border-radius:12px;display:grid;place-items:center;background:linear-gradient(145deg,#54e1d0,#397ec2);color:#06121e;font-weight:900;font-size:18px;box-shadow:0 8px 26px #2acbb52b}
.brand strong{font-size:14px;letter-spacing:.14em}.brand small{display:block;color:var(--muted);font-size:10px;letter-spacing:.18em;margin-top:1px}
.pill{display:inline-flex;gap:8px;align-items:center;padding:7px 11px;border:1px solid var(--line);border-radius:99px;color:var(--muted);font-size:12px;background:#0d1726}
.dot{width:7px;height:7px;background:var(--green);border-radius:50%;box-shadow:0 0 12px #45d6a088}
.hero{padding:30px 0 22px;display:flex;justify-content:space-between;align-items:end;gap:18px}.eyebrow{font-size:10px;letter-spacing:.2em;text-transform:uppercase;color:var(--teal);font-weight:700}
h1{font-size:clamp(27px,4vw,38px);line-height:1.15;margin:8px 0 9px;letter-spacing:-.035em}.sub{color:var(--muted);max-width:700px;margin:0}
.grid{display:grid;grid-template-columns:minmax(0,1.25fr) minmax(300px,.75fr);gap:18px;align-items:start}
.left,.right{display:grid;gap:18px;min-width:0}.card{background:linear-gradient(160deg,#121e30f2,#0e1828f5);border:1px solid var(--line);border-radius:17px;padding:20px;box-shadow:0 14px 42px #0000001c}
.card-head{display:flex;justify-content:space-between;align-items:center;gap:12px;margin-bottom:16px}.card-head h2{font-size:14px;letter-spacing:.01em;margin:0}.hint{color:var(--muted);font-size:11px}
.fields{display:grid;grid-template-columns:1fr 1fr;gap:13px}.field{min-width:0}.field.wide{grid-column:1/-1}label{display:block;color:#b5c4d7;font-size:11px;font-weight:650;margin-bottom:6px}
input,textarea{display:block;width:100%;min-width:0;color:var(--text);background:#091321;border:1px solid #2a3d56;border-radius:9px;padding:10px 11px;outline:none;transition:border .15s,box-shadow .15s}
input:focus,textarea:focus{border-color:#4ccfca;box-shadow:0 0 0 3px #4ccfca1a}input::placeholder,textarea::placeholder{color:#62758e}
input[type=checkbox]{width:16px;height:16px;accent-color:var(--teal)}textarea{min-height:86px;resize:vertical}
.inline-options{display:flex;flex-wrap:wrap;gap:15px 20px;padding-top:4px}.check{display:flex;align-items:center;gap:8px;color:#bac8d9;font-size:12px}
.rows{display:grid;gap:10px}.item{background:#0b1524;border:1px solid #263850;border-radius:12px;padding:13px;display:grid;gap:9px;min-width:0}
.item-top{display:flex;align-items:center;justify-content:space-between;gap:10px}.item-title{font-size:12px;font-weight:700;color:#d9e6f5}
.test-fields{display:grid;grid-template-columns:minmax(160px,.65fr) minmax(0,1.6fr);gap:9px}.test-fields .needs{grid-column:1/-1}
.btn{border:1px solid #344b67;border-radius:9px;background:#17263a;color:#dce9f7;padding:9px 12px;font-weight:650;font-size:12px;transition:.15s}.btn:hover{border-color:#4a7998;background:#1b3048}.btn.primary{border:0;color:#06151d;background:linear-gradient(110deg,#49ddc9,#6db4ff);box-shadow:0 8px 24px #48d5c533}.btn.primary:hover{filter:brightness(1.08)}.btn.ghost{background:transparent}.btn.remove{padding:5px 9px;color:#f19aa7;border-color:#693b4b;background:#281925}
.run{width:100%;min-height:52px;font-size:14px;letter-spacing:.01em}.run:disabled{opacity:.55;cursor:wait}
.metrics{display:grid;grid-template-columns:repeat(3,1fr);gap:9px}.metric{padding:13px 11px;border-radius:11px;background:#0a1422;border:1px solid #263850}.metric b{display:block;font-size:24px;line-height:1.2}.metric span{font-size:10px;color:var(--muted);text-transform:uppercase;letter-spacing:.1em}
.statusline{display:flex;justify-content:space-between;gap:12px;align-items:center;margin:12px 0;color:var(--muted);font-size:12px}.status{font-weight:750;letter-spacing:.08em}.status.PASS{color:var(--green)}.status.FAIL{color:var(--red)}.status.PARTIAL{color:var(--amber)}.status.RUNNING{color:var(--teal)}
.progress{height:5px;background:#1b2b40;border-radius:99px;overflow:hidden}.progress span{display:block;height:100%;width:0;background:linear-gradient(90deg,#43d6c5,#81adff);transition:width .3s}
.log-list{display:grid;gap:9px}.log{border:1px solid #263850;border-radius:10px;overflow:hidden;background:#0a1422}.log summary{list-style:none;cursor:pointer;display:flex;align-items:center;justify-content:space-between;gap:10px;padding:11px 12px;font-size:12px}.log summary::-webkit-details-marker{display:none}.log-body{padding:0 12px 12px}.log pre{white-space:pre-wrap;overflow-wrap:anywhere;color:#b4c4d8;font:11px/1.55 ui-monospace,SFMono-Regular,Consolas,monospace;background:#070e18;padding:11px;border-radius:8px;max-height:320px;overflow:auto}.tag{font-size:9px;letter-spacing:.1em;font-weight:800;padding:3px 7px;border-radius:5px;background:#1c3142;color:var(--blue)}.tag.PASS{background:#10372e;color:var(--green)}.tag.FAIL{background:#3a202b;color:var(--red)}.tag.SKIP{background:#3a3020;color:var(--amber)}
.note{font-size:11px;color:var(--muted);line-height:1.6}.download{display:flex;justify-content:center;margin-top:12px;text-decoration:none}.download.disabled{pointer-events:none;opacity:.4}
.footer{margin-top:18px;color:#6f829a;font-size:10px;text-align:center}
@media(max-width:900px){.grid{grid-template-columns:1fr}.right{grid-row:auto}.hero{align-items:start;flex-direction:column}}
@media(max-width:560px){.shell{padding:15px 12px 30px}.card{padding:15px;border-radius:14px}.fields,.test-fields{grid-template-columns:1fr}.field.wide,.test-fields .needs{grid-column:auto}.top{align-items:flex-start}.hero{padding:24px 0 18px}.metrics{gap:6px}.metric{padding:10px 8px}.metric b{font-size:20px}.metric span{font-size:9px}}
</style></head><body><div class="shell">
<header class="top"><div class="brand"><div class="mark">T</div><div><strong>TESSERA</strong><small>ENGINEERING · VALIDATION CENTER</small></div></div><div class="pill"><i class="dot"></i> COLAB RUNTIME CONNECTED</div></header>
<section class="hero"><div><div class="eyebrow">Portable test workspace</div><h1>Validation Console</h1><p class="sub">Configure your repository, compose setup steps and test suites, then execute them on the active Colab runtime.</p></div><div class="pill">● &nbsp; Private session route</div></section>
<div class="grid"><main class="left">
<section class="card"><div class="card-head"><h2>01 / Repository & runtime</h2><span class="hint">Checkout configuration</span></div>
<div class="fields">
<div class="field wide"><label for="repo">GitHub repository URL</label><input id="repo" value="https://github.com/Jue-dev/Tessera.git" autocomplete="off"></div>
<div class="field"><label for="ref">Branch / ref</label><input id="ref" value="issue-6/unwind-lifecycle"></div>
<div class="field"><label for="secret">Colab Secret name</label><input id="secret" value="GH_TOKEN" autocomplete="off"></div>
<div class="field"><label for="gituser">Git username</label><input id="gituser" value="x-access-token"></div>
<div class="field"><label for="depth">Clone depth (0 = full)</label><input id="depth" type="number" min="0" max="10000" value="1"></div>
<div class="field"><label for="cloneTimeout">Clone timeout · seconds</label><input id="cloneTimeout" type="number" min="1" max="3600" value="240"></div>
<div class="field"><label for="testTimeout">Step timeout · seconds</label><input id="testTimeout" type="number" min="1" max="7200" value="900"></div>
</div><div class="inline-options"><label class="check"><input id="keep" type="checkbox"> Keep cloned workspace</label><label class="check"><input id="continue" type="checkbox" checked> Continue after test failures</label></div>
</section>
<section class="card"><div class="card-head"><h2>02 / Setup pipeline</h2><button class="btn" id="addSetup">＋ Add setup step</button></div><div class="rows" id="setupRows"></div><p class="note">Optional shell commands run after checkout and before tests. Commands run on the Colab server, not in this browser tab.</p></section>
<section class="card"><div class="card-head"><h2>03 / Test matrix</h2><button class="btn" id="addTest">＋ Add test</button></div><div class="rows" id="testRows"></div><p class="note">Add, remove, rename, and reorder tests by editing their rows. Tool requirements are checked before execution.</p></section>
</main><aside class="right">
<section class="card"><div class="card-head"><h2>Execution control</h2><span class="tag" id="runtimeTag">READY</span></div>
<button id="run" class="btn primary run">▶ &nbsp; Run validation pipeline</button>
<div class="statusline"><span id="stage">Ready to configure</span><b id="overall" class="status">IDLE</b></div><div class="progress"><span id="progress"></span></div>
<div class="metrics" style="margin-top:14px"><div class="metric"><b id="passed">0</b><span>Passed</span></div><div class="metric"><b id="failed">0</b><span>Failed</span></div><div class="metric"><b id="skipped">0</b><span>Skipped</span></div></div>
<a id="download" class="btn download disabled" href="#" target="_blank" download>↓ &nbsp; Download validation report</a>
</section>
<section class="card"><div class="card-head"><h2>Execution log</h2><span class="hint" id="logCount">0 steps</span></div><div id="logs" class="log-list"><div class="note">Execution output will appear here after the pipeline starts.</div></div></section>
<section class="card"><div class="card-head"><h2>Runtime security</h2><span class="tag">SESSION</span></div><p class="note">The GitHub token is read from Colab Secrets for checkout only. It is not sent to this page or included in the report; credential variables are removed before setup and test commands. Colab Secrets are not isolated from untrusted code running in the same runtime. Execute trusted repositories and commands only.</p></section>
</aside></div><div class="footer">TESSERA VALIDATION CENTER · SESSION-SCOPED COLAB SERVER · DARK INTERFACE</div></div>
<script>
const BASE='__BASE_PATH__';
const $=id=>document.getElementById(id);
let busy=false;
const defaults=[
{name:'GCC strict mock',command:'bash verify/windows_mock/run_windows_mock_tests.sh',needs:''},
{name:'Clang strict mock',command:"EXTRA_CXXFLAGS='-stdlib=libc++' CXX=clang++ bash verify/windows_mock/run_windows_mock_tests.sh",needs:'clang++'},
{name:'GCC ASan + UBSan',command:"EXTRA_CXXFLAGS='-fsanitize=address,undefined -fno-omit-frame-pointer' ASAN_OPTIONS='detect_leaks=1:halt_on_error=1' UBSAN_OPTIONS='halt_on_error=1:print_stacktrace=1' bash verify/windows_mock/run_windows_mock_tests.sh",needs:''}
];
function node(tag,cls,text){const n=document.createElement(tag);if(cls)n.className=cls;if(text!==undefined)n.textContent=text;return n}
function addSetup(value=''){
 const box=node('div','item');const top=node('div','item-top');top.append(node('span','item-title','Setup command'));
 const remove=node('button','btn remove','Remove');remove.type='button';remove.onclick=()=>box.remove();top.append(remove);
 const input=node('input');input.className='setup-command';input.placeholder='Trusted setup command';input.value=value;box.append(top,input);$('setupRows').append(box);
}
function addTest(item={}){
 const box=node('div','item');const top=node('div','item-top');top.append(node('span','item-title','Test definition'));
 const remove=node('button','btn remove','Remove');remove.type='button';remove.onclick=()=>box.remove();top.append(remove);box.append(top);
 const fields=node('div','test-fields');const name=node('input');name.className='test-name';name.placeholder='Test name';name.value=item.name||'';
 const cmd=node('textarea');cmd.className='test-command';cmd.placeholder='Shell command to execute';cmd.value=item.command||'';
 const needs=node('input');needs.className='needs';needs.placeholder='Required tools, comma-separated (optional)';needs.value=item.needs||'';
 fields.append(name,cmd,needs);box.append(fields);$('testRows').append(box);
}
$('addSetup').onclick=()=>addSetup();$('addTest').onclick=()=>addTest();
addSetup();defaults.forEach(addTest);
async function request(path,options={}){
 const response=await fetch(BASE+path,{cache:'no-store',...options});
 const data=await response.json().catch(()=>({error:'Invalid server response'}));
 if(!response.ok)throw new Error(data.error||('HTTP '+response.status));
 return data;
}
function renderLogs(items){
 const root=$('logs');root.replaceChildren();$('logCount').textContent=items.length+' steps';
 if(!items.length){root.append(node('div','note','No execution output yet.'));return}
 items.slice().reverse().forEach(item=>{
  const d=node('details','log');const s=node('summary');const name=node('span','',item.name);const tag=node('span','tag '+item.status,item.status+' · '+item.duration+'s');s.append(name,tag);d.append(s);
  const body=node('div','log-body');const command=node('div','note','COMMAND: '+item.command);const pre=node('pre','',item.output||'(no output)');body.append(command,pre);d.append(body);root.append(d);
 });
}
function renderState(s){
 $('stage').textContent=s.stage||'Ready';$('overall').textContent=s.overall||'IDLE';$('overall').className='status '+(s.overall||'');
 $('passed').textContent=s.passed||0;$('failed').textContent=s.failed||0;$('skipped').textContent=s.skipped||0;
 $('runtimeTag').textContent=s.running?'RUNNING':(s.overall||'READY');
 $('progress').style.width=s.running?'72%':(s.report_ready?'100%':'0%');
 $('run').disabled=!!s.running;$('run').textContent=s.running?'◌  Validation in progress…':'▶  Run validation pipeline';
 $('download').classList.toggle('disabled',!s.report_ready);if(s.report_ready)$('download').href=BASE+'api/report';
 renderLogs(s.logs||[]);
}
async function poll(){
 try{const s=await request('api/state');renderState(s);busy=!!s.running;if(s.running)setTimeout(poll,1800)}
 catch(e){$('stage').textContent='Server connection lost: '+e.message;$('runtimeTag').textContent='OFFLINE';setTimeout(poll,3500)}
}
$('run').onclick=async()=>{
 if(busy)return;
 const setup=[...document.querySelectorAll('.setup-command')].map(x=>x.value.trim()).filter(Boolean);
 const tests=[...document.querySelectorAll('#testRows .item')].map(x=>({
  name:x.querySelector('.test-name').value.trim(),
  command:x.querySelector('.test-command').value.trim(),
  needs:x.querySelector('.needs').value.trim()
 }));
 if(tests.some(x=>!x.name||!x.command)){alert('Every test row needs both a name and a command.');return}
 const cfg={repo:$('repo').value,ref:$('ref').value,secret_name:$('secret').value,git_user:$('gituser').value,
  depth:Number($('depth').value),clone_timeout:Number($('cloneTimeout').value),test_timeout:Number($('testTimeout').value),
  keep_workspace:$('keep').checked,continue_on_error:$('continue').checked,setup,tests};
 try{busy=true;await request('api/run',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(cfg)});poll()}
 catch(e){busy=false;alert(e.message)}
};
poll();
</script></body></html>'''

class TesseraHandler(BaseHTTPRequestHandler):
    server_version = "TesseraConsole"
    protocol_version = "HTTP/1.0"

    def log_message(self, *args):
        pass

    def send_bytes(self, code, body, content_type="application/json; charset=utf-8", extra=None):
        self.send_response(code)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store, max-age=0")
        self.send_header("Pragma", "no-cache")
        self.send_header("x-colab-notebook-cache-control", "no-cache")
        self.send_header("X-Content-Type-Options", "nosniff")
        self.send_header("Referrer-Policy", "no-referrer")
        if extra:
            for key, value in extra.items():
                self.send_header(key, value)
        self.end_headers()
        self.wfile.write(body)

    def send_json(self, code, value):
        self.send_bytes(code, json.dumps(value, ensure_ascii=False).encode("utf-8"))

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == BASE_PATH:
            body = PAGE.replace("__BASE_PATH__", BASE_PATH).encode("utf-8")
            self.send_bytes(200, body, "text/html; charset=utf-8")
        elif path == BASE_PATH + "api/state":
            with LOCK:
                snapshot = dict(STATE)
                snapshot["logs"] = list(STATE["logs"])
            self.send_json(200, snapshot)
        elif path == BASE_PATH + "api/report":
            if not REPORT_PATH.exists():
                self.send_json(404, {"error": "No report is available yet."})
            else:
                payload = REPORT_PATH.read_bytes()
                self.send_bytes(200, payload, "text/plain; charset=utf-8", {
                    "Content-Disposition": 'attachment; filename="PROJECT_VALIDATION_REPORT.txt"'
                })
        else:
            self.send_json(404, {"error": "Not found"})

    def do_POST(self):
        path = urlsplit(self.path).path
        if path != BASE_PATH + "api/run":
            self.send_json(404, {"error": "Not found"})
            return
        try:
            length = int(self.headers.get("Content-Length", "0"))
            if length <= 0 or length > 600000:
                raise ValueError("Invalid request size.")
            config = json.loads(self.rfile.read(length).decode("utf-8"))
            if not isinstance(config, dict):
                raise ValueError("Invalid configuration.")
            with LOCK:
                if STATE["running"]:
                    self.send_json(409, {"error": "A validation job is already running."})
                    return
                STATE.update(running=True, stage="Queued", overall="RUNNING",
                             logs=[], passed=0, failed=0, skipped=0, report_ready=False)
            threading.Thread(target=run_validation, args=(config,), daemon=True).start()
            self.send_json(202, {"accepted": True})
        except Exception as exc:
            self.send_json(400, {"error": str(exc)})

# Restart cleanly if this launch cell is executed again in the same runtime.
_previous = globals().get("_TESSERA_HTTPD")
if _previous is not None:
    try:
        _previous.shutdown()
        _previous.server_close()
    except Exception:
        pass

httpd = ThreadingHTTPServer(("0.0.0.0", 0), TesseraHandler)
httpd.daemon_threads = True
PORT = httpd.server_address[1]
threading.Thread(target=httpd.serve_forever, name="tessera-http", daemon=True).start()
_TESSERA_HTTPD = httpd

print("Tessera validation server is running inside this Colab runtime.")
print("Port:", PORT)
print("Session route:", BASE_PATH)
if colab_output is None:
    print("Colab port proxy is unavailable in this runtime.")
else:
    try:
        proxy_base = colab_output.eval_js("google.colab.kernel.proxyPort(" + str(PORT) + ", {cache: false})")
        console_url = str(proxy_base).rstrip("/") + BASE_PATH
        safe_url = html.escape(console_url, quote=True)
        display(HTML(
            '<div style="background:#101a2b;border:1px solid #2b415c;border-radius:14px;'
            'padding:18px;color:#edf3fb;font:14px system-ui;max-width:900px">'
            '<div style="color:#43d6c5;font-size:11px;font-weight:700;letter-spacing:.16em;margin-bottom:8px">TESSERA · VALIDATION SERVER READY</div>'
            '<a href="' + safe_url + '" target="_blank" rel="noopener noreferrer" '
            'style="display:inline-block;background:linear-gradient(110deg,#49ddc9,#6db4ff);'
            'color:#06151d;text-decoration:none;font-weight:800;padding:12px 18px;border-radius:9px">'
            'OPEN STANDALONE CONSOLE ↗</a>'
            '<div style="margin-top:12px;color:#91a3ba;font-size:11px">Open in a separate browser tab. This session URL expires when the Colab runtime ends.</div>'
            '<div style="margin-top:8px;overflow-wrap:anywhere;color:#b9c9dc;font:11px ui-monospace,monospace">'
            + safe_url + '</div></div>'
        ))
        print("Standalone console URL:", console_url)
    except Exception as exc:
        print("Could not generate the Colab proxy URL:", type(exc).__name__, str(exc))
        print("The local server is running, but Colab did not return an openable proxy URL.")
